# NALAPRO Final Project - Naa Lamiorkor Boye

## Installing and Importing Libraries and Dataset

In [2]:
!pip install numpy pandas matplotlib scikit-learn nltk gensim torch transformers datasets

In [16]:
# Core libraries
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NLTK
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

# Sklearn
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Gensim
from gensim.models import Word2Vec

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from collections import Counter


# Transformers
# from transformers import (
#     BertTokenizer,
#     BertForSequenceClassification,
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     Trainer,
#     TrainingArguments
# )

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/LamiorkorBoye/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/LamiorkorBoye/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Preliminary Look at Data

In [6]:
# Storing the training and testing sets of the data, and setting X and y variables   
train_data = fetch_20newsgroups(subset='train')
test_data = fetch_20newsgroups(subset='test')

X_train = train_data.data
y_train = train_data.target

X_test = test_data.data
y_test = test_data.target

In [7]:
# Displaying key information about the dataset

print("Number of training documents:", len(X_train))
print("Number of test documents:", len(X_test))
print("Number of classes:", len(train_data.target_names))
print("Class names:", train_data.target_names)

Number of training documents: 11314
Number of test documents: 7532
Number of classes: 20
Class names: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [9]:
# Displaying a sample document from the training set

print("\nSample document:\n")
print(X_train[0])


Sample document:

From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----







In [11]:
# Displaying the label and category of the sample document

print("\nLabel index:", y_train[0])
print("Category:", train_data.target_names[y_train[0]])


Label index: 7
Category: rec.autos


In [13]:
# Displaying the distribution of classes in the training set

print("\nClass distribution:\n")
label_counts = Counter(y_train)
for label, count in sorted(label_counts.items()):
    print(train_data.target_names[label], ":", count)


Class distribution:

alt.atheism : 480
comp.graphics : 584
comp.os.ms-windows.misc : 591
comp.sys.ibm.pc.hardware : 590
comp.sys.mac.hardware : 578
comp.windows.x : 593
misc.forsale : 585
rec.autos : 594
rec.motorcycles : 598
rec.sport.baseball : 597
rec.sport.hockey : 600
sci.crypt : 595
sci.electronics : 591
sci.med : 594
sci.space : 593
soc.religion.christian : 599
talk.politics.guns : 546
talk.politics.mideast : 564
talk.politics.misc : 465
talk.religion.misc : 377


## Pre-Processing the Data

In [15]:
# Preprocessing the data by removing headers, footers, and quotes

train_data = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
)

test_data = fetch_20newsgroups(
    subset='test',
    remove=('headers', 'footers', 'quotes')
)

X_train = train_data.data
y_train = train_data.target
X_test = test_data.data
y_test = test_data.target

In [17]:
# Preprocessing function to clean and tokenize the text data

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # lowercase
    text = text.lower()
    
    # remove numbers
    text = re.sub(r'\d+', ' ', text)
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # tokenize
    tokens = word_tokenize(text)
    
    # keep only alphabetic words and remove stopwords
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    
    return tokens

In [ ]:
# Preprocessing the training and testing data

X_train_tokens = [preprocess_text(doc) for doc in X_train]
X_test_tokens = [preprocess_text(doc) for doc in X_test]

In [26]:
# Displaying the original and preprocessed versions of a sample document

print("Original text:")
print(X_train[0][:500])

print("\nTokenized and cleaned (First 10 tokens):")
print(X_train_tokens[0][:10])

Original text:
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Tokenized and cleaned (First 10 tokens):
['wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car']


In [ ]:
# Checking for empty documents after preprocessing

empty_train = sum(1 for doc in X_train_tokens if len(doc) == 0)
empty_test = sum(1 for doc in X_test_tokens if len(doc) == 0)

print("Empty training documents:", empty_train)
print("Empty test documents:", empty_test)

Empty training documents: 314
Empty test documents: 230


In [30]:
# Creating cleaned text strings for TF-IDF later

X_train_clean = [' '.join(tokens) for tokens in X_train_tokens]
X_test_clean = [' '.join(tokens) for tokens in X_test_tokens]

## Word2Vec

In [ ]:
# Training a Word2Vec model on the tokenized training data
word2vec_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

# Displaying some information about the trained Word2Vec model
print("Vocabulary size:", len(word2vec_model.wv))

if 'car' in word2vec_model.wv:
    print("Vector for 'car':", word2vec_model.wv['car'][:10])
else:
    print("'car' is not in the vocabulary.")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary size: 41086
Vector for 'car': [ 0.00149113 -0.07019585  0.22644977  0.23866768  0.3906848   0.3269034
 -0.12027666  0.35119155 -0.31846008 -0.54287547]


In [35]:
# Exploring the most similar words to some key terms in the Word2Vec model

print(word2vec_model.wv.most_similar("car", topn=10))
print(word2vec_model.wv.most_similar("computer", topn=10))
print(word2vec_model.wv.most_similar("hockey", topn=10))

[('volvo', 0.7346950173377991), ('cars', 0.7258910536766052), ('dealership', 0.6956921219825745), ('taurus', 0.6941965222358704), ('porsche', 0.6930862665176392), ('convertible', 0.6867547035217285), ('getaway', 0.6799867749214172), ('truck', 0.6791388392448425), ('aftermarket', 0.6754514575004578), ('gtz', 0.669730544090271)]
[('dgree', 0.6528011560440063), ('usnos', 0.642188310623169), ('computers', 0.6313028931617737), ('architecture', 0.6211101412773132), ('aided', 0.6154389977455139), ('shopper', 0.609842836856842), ('doublemajor', 0.5990592241287231), ('programmer', 0.598799467086792), ('networks', 0.594516396522522), ('mainframe', 0.5893483757972717)]
[('nhl', 0.7704172134399414), ('championship', 0.7247042059898376), ('allstar', 0.7049854397773743), ('teams', 0.7007152438163757), ('petes', 0.6972623467445374), ('basketball', 0.6949005722999573), ('ncaa', 0.6934357285499573), ('rosters', 0.6911771893501282), ('tournament', 0.691114068031311), ('playoff', 0.6907896399497986)]


In [36]:
# Function to create document vectors by averaging the word vectors of the tokens in the document

def document_vector(model, doc_tokens):
    vectors = []
    
    for word in doc_tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

In [37]:
# Creating document vectors for the training and testing sets

X_train_vec = np.array([document_vector(word2vec_model, doc) for doc in X_train_tokens])
X_test_vec = np.array([document_vector(word2vec_model, doc) for doc in X_test_tokens])

print(X_train_vec.shape)
print(X_test_vec.shape)

(11314, 100)
(7532, 100)
